# Install cyvcf2

In [1]:
!pip install cyvcf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.3/828.3 kB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [cyvcf2]━━━━ 1/4 [click]


# Import

In [2]:
from cyvcf2 import VCF

In [13]:
import pandas as pd

In [34]:
import sys

In [31]:
yes = 0
no = 0
vars_to_check = []
for variant in VCF("../data/sv_output/sniffles/filtered/HG00290.merged.sniffles.vcf"):
    if len(variant.ALT[0]) > 5:
        row = {
            "Chrom": variant.CHROM,
            "Pos": variant.POS,
            "Qual": variant.QUAL,
            "Alt1": variant.ALT[0],
            "Len1": len(variant.ALT[0])
        }
        vars_to_check.append(row)

In [32]:
var_df = pd.DataFrame(vars_to_check)

In [33]:
var_df

,Chrom,Pos,Qual,Alt1,Len1
0,chr1,136963,60.0,GTGGGAGGGGCCGGTGTGAGACAAGGGCTCAGGCTGACCTCTCTCA...,145
1,chr1,157348,60.0,CTACTCAGGAGGCTGAGGCAGGAGAATCGCTTGAACCTGGGAGGCA...,144
2,chr1,181146,50.0,CCGCCGGCGCAGGCGCAGAGAGGCGCGCCGCGCCGGCGCAGGCGCA...,117
3,chr1,191372,50.0,CAGTCACATCCCTTTAAACTGGATCCACACAGTAAGAGGACTCTGA...,2001
4,chr1,204560,52.0,ATGGGATTGCTGGGTCAAATGGTATTTCTAGTTCTAGATCCTTGAG...,3069
...,...,...,...,...,...
13873,chrY,56845781,55.0,TTATGTGATGTAACACGTTTATAAGCACTGCCTACAGGGAATTTTG...,4899
13874,chrY,56851299,59.0,ATGACATATCTCTGCACTGATCACCCCAGGGAGAGAATTCTTGTTT...,70
13875,chrY,56856795,58.0,TATATTAAAATTTCAATCAACAGTCACAAAAGCAGACTAATAAAGC...,61
13876,chrY,56867594,58.0,TCCTGCCTCAGCCTCCCAAGTAGCTGGGATTACAGGCACCTGCCGC...,134


# Kmers

## Modified kc-py1
### https://github.com/lh3/kmer-cnt/blob/master/kc-py1.py

In [43]:
base_for = "ACGT"
base_rev = "TGCA"
comp_tab = str.maketrans(base_for, base_rev)

In [35]:
def count_kmer(h, k, seq):
	l = len(seq)
	if l < k: return
	for i in range(l - k + 1):
		kmer_for = seq[i:(i+k)]
		if 'N' in kmer_for: continue
		kmer_rev = kmer_for.translate(comp_tab)[::-1]
		if kmer_for < kmer_rev: kmer = kmer_for
		else: kmer = kmer_rev
		if kmer in h:
			h[kmer] += 1
		else: h[kmer] = 1

In [119]:
def count_stdin(k,seqs):
	counter = {}
	seq = []
	for line in seqs:
		if line[0] == '>':
			if len(seq) > 0:
				count_kmer(counter, k, ''.join(seq))
				seq = []
		else:
			seq.append(line[:-1])
	if len(seq) > 0:
		count_kmer(counter, k, ''.join(seq).upper())
	return counter

In [180]:
def print_hist(counter):
    hist_list = []
    hist = [0] * 256
    for kmer in counter:
        cnt = counter[kmer]
        if cnt > 255: cnt = 255
        hist[cnt] += 1
    for i in range(1, 256):
        if hist[i] > 0:
            hist_list.append([i, hist[i]])
    print(hist_list)

## Test kmer counting

In [87]:
alts = var_df["Alt1"].tolist()

In [ ]:
[alts[5]]

In [95]:
24*47 + 142

1270

In [93]:
len(alts[5])

1317

In [90]:
len("CTGCGACACTCACGCGGGTGCCATCTCAGCAGCTCACGGTGTGGAAA")

47

In [150]:
counter = count_stdin(48,[alts[5]])

In [151]:
print_hist(counter)

1	144
23	3
24	44


In [109]:
print(alts[45])

TCTTTCCTTCCCTTTCCCTCCCTCCCTTCCTTCCTCTTTCCTTCCTTCCTTTCCCTCCCTTACTCCTTCCTTCCTTCCCTTCCCCTTCCTTCTTCCTTCTCTCCCTCCCTC


In [117]:
test_seq = "GCC"*50

In [118]:
test_seq

'GCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCC'

In [ ]:
for i in range(1,len(test_seq)//2):
    counter = count_stdin(i,[test_seq])
    print(i)
    print_hist(counter)

# Compressibility

### Following a simialr method to this: https://www.nature.com/articles/s41598-022-17267-z

In [191]:
import zlib

In [192]:
test_seq

'GCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCC'

In [193]:
sys.getsizeof(test_seq)

191

In [195]:
len(test_seq.encode('utf-8'))

150

In [197]:
len(zlib.compress(test_seq.encode('utf-8')))

14

In [205]:
var_df['uncomp'] = var_df['Alt1'].str.encode(encoding="utf-8").str.len()

In [209]:
var_df['comp'] = var_df['Alt1'].apply(lambda x: zlib.compress(x.encode('utf-8'))).str.len()

In [211]:
var_df['ratio'] = var_df['uncomp']/var_df['comp']

In [219]:
df_sorted = var_df.sort_values(by=['ratio'],ascending=False,ignore_index=True)

In [220]:
df_sorted

,Chrom,Pos,Qual,Alt1,Len1,uncomp,comp,ratio
0,chr14,23280710,60.0,TTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTAT...,800,800,21,38.095238
1,chr11,126065033,60.0,TATAGTATTAAGAGGTGATAGTATTAAGAGGTGATAGTATTAAGAG...,1894,1894,51,37.137255
2,chr1,248408457,60.0,TAGATAGCGGCAGATGGTCACGGGAGTTTAGCTCGGGCTAGAGCAT...,35414,35414,961,36.851197
3,chr16,7898322,60.0,CTGTATATATATCTATATATACATATATATCTATATACTGTATATA...,1692,1692,51,33.176471
4,chr5,176591787,60.0,GATGGTGTTGGTGTTGAGGTGATGGTGATGGTTGTGATGGTGATGG...,1645,1645,50,32.900000
...,...,...,...,...,...,...,...,...
13873,chr21,46140360,60.0,CCCCCGTCCACCTTCACGTGGCTCACAGGGAGTGTAAACCAATCCA...,51,51,42,1.214286
13874,chr20,64065883,60.0,AAGTTTGGCCTCAGCTCGGGACACTGATGTTCCCGGGTTGGCCCTT...,52,52,43,1.209302
13875,chr19,55308312,60.0,CCTGTACCTGGCTCCAACTTATTTTTCTTACATTTCTGGTCCTTCA...,52,52,43,1.209302
13876,chr10,174940,60.0,ATTGGTCAGAACCCATGGGCACTACAGTGGCACATGCTTGTCATTC...,51,51,43,1.186047


In [221]:
print(df_sorted['Alt1'][0])

TTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTATTTTA


In [223]:
print(df_sorted['Alt1'].tolist()[-1])

GTCTTATGGGTGATTATGTCATGACACAAAGTGTTCTCCTGATCTTATCTC
